# Stage 7 -- 细胞通讯分析 (Cell Communication: Ligand-Receptor Analysis)

本 notebook 做细胞间配体-受体通讯分析，识别不同细胞类型之间
通过分泌配体-表面受体进行"跨细胞对话"的信号通路：

1. **CellChat**（R，subprocess Rscript + 守卫）— 基于 curated 配体-受体数据库的
   通讯网络推断
2. **CellPhoneDB**（Python，守卫跳过）— 基于统计检验的配体-受体显著性分析
3. **配体-受体表达概览**（纯 Python，不依赖外部工具）— 基于表达矩阵的简单
   LR dotplot，工具不可用时仍可产出

## 生物学背景

**为什么做细胞通讯分析？** 单细胞数据提供了组织中所有细胞类型的
"分子快照"，但组织功能不是细胞各自独立表达基因就能实现的。
细胞通过**配体-受体 (ligand-receptor, LR) 相互作用**进行跨细胞通讯——
一个细胞分泌配体（如生长因子、趋化因子、细胞因子），另一个细胞
表达对应受体，接收信号后改变自身行为。

例如，在胃"炎-癌"转化微环境中：
- 上皮细胞分泌 EGF/TGF-alpha 激活成纤维细胞的 EGFR 通路
- 免疫细胞分泌 IL-1beta/TNF 诱导上皮细胞的 NF-kB 促炎响应
- 成纤维细胞分泌 WNT 配体维持上皮干细胞 niche

从 scRNA-seq 数据重建这些跨细胞通讯网络，能回答以下问题：
1. **哪些细胞类型是关键的信号发送者/接收者？**
2. **哪些信号通路在特定生物学过程中被激活？**
3. **疾病 vs 正常条件下通讯网络如何重塑？**

## 工具对比

| 工具 | 语言 | 核心思路 | 配体-受体数据库 |
|------|------|----------|-----------------|
| **CellChat** | R | 基于质量作用模型 + 网络分析，推断通讯概率 | CellChatDB (curated, ~2000+ LR pairs) |
| **CellPhoneDB** | Python | 基于置换检验的统计显著性 | CellPhoneDB (curated，含多亚基复合物) |

**为什么用两种工具？** CellChat 擅长网络层面的分析（识别主导信号通路、
通讯模式聚类），CellPhoneDB 擅长统计显著性检验和
多亚基受体复合物的精确建模。两者互补。

## 工具依赖与当前状态

- **CellChat** 是 R 包，需单独安装 R + CellChat R 包
- **CellPhoneDB** 是 Python 包，需 `pip install cellphonedb`
- 本 notebook 在工具不可用时**优雅跳过**对应步骤，并提供清晰的安装指引
- **纯 Python 的 LR 表达概览**不依赖任何外部工具，始终可运行

产出：
- CellChat 通讯网络 → `results/tables/stage7_cellchat_*`
- CellPhoneDB 显著 LR pairs → `results/tables/stage7_cpdb_*`
- 可视化 → `results/figures/stage7_cellcomm_*`


## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：Stage 6（细胞注释），读 `stage6_annotated_v*.h5ad`，读对应 `.h5ad` 文件
- **下游**：本 stage 产出 checkpoint 供 stage7 综合报告或后续可视化使用

### 为什么要迭代回跑？
细胞通讯分析的结果取决于分组列（`GROUP_COL`）和 LR 数据库的选择。
如果在检查通讯网络结果时发现：
- 通讯网络过于稀疏或杂乱 → 检查 `GROUP_COL` 是否为合适的细胞类型
  （当前默认 `leiden_res_0.6`，PI 完成注释后应改为 `cell_type_final_v1`）
- CellChat 通讯概率过低 → 确保数据中有 counts layer（原始计数而非 log-normalized），
  CellChat 的 over-expression 检测需要原始分布
- CellPhoneDB 发现过多假阳性 → 调低 `CPDB_PVAL_THRESHOLD`

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `nancang_stage6_annotated_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `nancang_stage7_cell_communication_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改分析参数）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为结果质量可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：后续 stage 的 `UPSTREAM_PATH` 指向你决定采用的版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"stage7_cell_communication"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询"stage 7 有哪些版本？哪些依赖 stage6_v1？"，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

# === PARAMS ===

本 notebook 所有可调参数集中在此 cell。PI 修改后重跑即可。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH              -- stage6 注释结果 h5ad
# OUTPUT_PATH                -- 本 notebook 产出 checkpoint
# GROUP_COL                  -- 用于定义细胞分组的 obs 列
#                               当前默认 leiden_res_0.6（cell_type_final_v1 全 NaN）
#                               待 stage6 完成注释后 PI 可改为 cell_type_final_v1
# RSCRIPT_BIN                -- Rscript 可执行文件路径
# CELLCHAT_DB_SPECIES        -- CellChat 数据库物种（human / mouse）
# CELLCHAT_WORK_DIR          -- CellChat Rscript 临时工作目录
# CPDB_PVAL_THRESHOLD        -- CellPhoneDB 显著性阈值
# N_TOP_LR                   -- 可视化中展示的 top LR pairs 数量
# VIS_MIN_CELLS              -- 聚类簇最少细胞数（低于此数跳过，避免小簇统计不可靠）

UPSTREAM_PATH = "results/nancang_stage6_annotated_v1.h5ad"
OUTPUT_PATH   = "results/stage7_cell_communication_v1.h5ad"  # 版本号 _v1 与 adata.uns["version"] 保持一致；回跑时 bump _v1->v2

GROUP_COL = "leiden_res_0.6"
# 重要说明：cell_type_final_v1 当前全为 NaN，
# 因此默认使用 leiden_res_0.6 作为分组。
# PI 在完成 stage6 注释后将此参数改为 "cell_type_final_v1" 即可。

# RSCRIPT_BIN 通过 platform 模块统一解析（ADR-0010）
# rscript_bin() 找不到 Rscript 时会抛 RuntimeError，
# 用 try-except 优雅降级：设 RSCRIPT_BIN=None 让后续 CellChat 步骤跳过。
from scrna_integration.platform import rscript_bin
try:
    RSCRIPT_BIN = rscript_bin()
except RuntimeError:
    RSCRIPT_BIN = None
CELLCHAT_DB_SPECIES  = "human"
CELLCHAT_WORK_DIR    = "results/_cellchat_tmp"

CPDB_PVAL_THRESHOLD  = 0.05
N_TOP_LR             = 20
VIS_MIN_CELLS        = 10


In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/stage7/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc
_root = os.getcwd()
_root_candidates = [
    _root,
    os.path.abspath(os.path.join(_root, "..")),
    os.path.abspath(os.path.join(_root, "..", "..")),
]

for _cand in _root_candidates:
    if os.path.isdir(os.path.join(_cand, "src", "scrna_integration")):
        _root = _cand
        break
else:
    _root = os.environ.get("PROJECT_ROOT", _root)

if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs(CELLCHAT_WORK_DIR, exist_ok=True)
print(f"PROJECT_ROOT: {_root}")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")


In [ ]:
# 导入依赖。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import subprocess
import shutil
import warnings

from scipy.sparse import csr_matrix
from scipy.io import mmwrite

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="anndata")
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  numpy {np.__version__}")

# 标志变量初始化 —— 各功能 cell 在成功时赋 True。
_cellchat_done = False
_cpdb_done = False
_lr_overview_done = False


In [ ]:
# 加载上游 stage6 输出。
# 契约：需包含 GROUP_COL 列。
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# 检查分组列
print(f"GROUP_COL '{GROUP_COL}' 存在: {GROUP_COL in adata.obs.columns}")
if GROUP_COL in adata.obs.columns:
    _group_vals = adata.obs[GROUP_COL]
    # NaN 提示：cell_type_final_v1 默认全 NaN，
    # 此时应使用 leiden_res_0.6
    _n_nan = _group_vals.isna().sum()
    _n_unique = _group_vals.dropna().nunique() if _n_nan > 0 else _group_vals.nunique()
    if _n_nan > 0 and _n_unique == 0:
        print(f"WARNING: {GROUP_COL} 中 {_n_nan}/{len(_group_vals)} 个细胞为 NaN，"
              f"无有效分组！\n"
              f"  请将 PARAMS 中的 GROUP_COL 改为 'leiden_res_0.6' 或"
              f"先完成 stage6 细胞注释。")
    else:
        print(f"  分组数: {_n_unique}（{_n_nan} NaN 细胞）")

# 检查 embedding
print(f"X_umap: {'X_umap' in adata.obsm}")
print(f"layers: {list(adata.layers.keys())}")
print(f"obsm keys: {list(adata.obsm.keys())}")

# 过滤有效分组——排除 NaN 细胞、排除小簇
_group_raw = adata.obs[GROUP_COL].astype(str)
_group_counts = _group_raw.value_counts()
_valid_groups = _group_counts[_group_counts >= VIS_MIN_CELLS].index.tolist()
_skip_groups = _group_counts[_group_counts < VIS_MIN_CELLS].index.tolist()
if _skip_groups:
    # 排除 "nan" 字面值（NaN 转 str 的产物）
    _skip_groups = [g for g in _skip_groups if g != "nan"]
print(f"有效分组 (>= {VIS_MIN_CELLS} 细胞): {len(_valid_groups)}")
print(f"跳过分组 (< {VIS_MIN_CELLS} 细胞): {_skip_groups}")


## 工具可用性守卫

在运行 CellChat 和 CellPhoneDB 之前，检查：
1. **Rscript** 是否在 PATH 中，以及 CellChat R 包是否已安装
2. **CellPhoneDB Python 包**是否可导入

根据 grn.ipynb 确立的"无条件检查 + 合并消息"模式：
两个检查始终执行，即使第一个失败也继续检查第二个，
确保 PI 看到完整的依赖状态报告而非逐个排查。

任一条件不满足，对应步骤将优雅跳过并给出清晰安装指引。
本 notebook 的纯 Python LR 表达概览不依赖任何外部工具，始终可运行。

In [ ]:
# === 工具可用性检查 ===
# 同时检查 CellChat (R) + CellPhoneDB (Python)，
# 两个检查无条件执行，合并报告所有缺失项。

# ---- 1. CellChat R 环境检查 ----
# RSCRIPT_BIN 来自 PARAMS cell，已通过 try-except rscript_bin() 赋值；
# None 表示 Rscript 不可用，非 None 表示可用路径。
_r_available = RSCRIPT_BIN is not None
_cellchat_r_available = False
_cellchat_r_error = ""
print(f"Rscript 可用: {_r_available}  (RSCRIPT_BIN={RSCRIPT_BIN})")

if _r_available:
    _check_cmd = [
        RSCRIPT_BIN, "--vanilla", "-e",
        "suppressPackageStartupMessages(library(CellChat)); cat('OK')",
    ]
    try:
        _res = subprocess.run(
            _check_cmd, capture_output=True, text=True, timeout=60,
        )
        if _res.returncode == 0 and "OK" in _res.stdout:
            _cellchat_r_available = True
            print("CellChat R 包可用 -- CellChat 通讯分析将正常执行")
        else:
            _cellchat_r_error = (_res.stderr or _res.stdout or "").strip()[:300]
            print("CellChat R 包不可用（load 失败或未安装）")
    except (FileNotFoundError, subprocess.TimeoutExpired) as _e:
        _cellchat_r_error = str(_e)
        print(f"Rscript 调用失败: {_e}")
else:
    _cellchat_r_error = "Rscript 未在 PATH 中找到"

# ---- 2. CellPhoneDB Python 包检查 ----
_cpdb_available = False
_cpdb_error = ""
try:
    import cellphonedb  # noqa: F401
    _cpdb_available = True
    print("CellPhoneDB Python 包可用 -- 将运行统计显著性分析")
except ImportError as _e:
    _cpdb_error = str(_e)
    print(f"CellPhoneDB 不可用: {_e}")

# ---- 3. 综合判断与合并消息 ----
_cellchat_ready = _r_available and _cellchat_r_available
_any_skipped = not _cellchat_ready or not _cpdb_available

if not _cellchat_ready and not _cpdb_available:
    _skip_msg = (
        "=" * 60 + "\n"
        "CellChat (R) 和 CellPhoneDB (Python) 均不可用，\n"
        "这两个工具对应的步骤将跳过。\n\n"
        "纯 Python 的配体-受体表达概览不受影响，仍可正常产出。\n\n"
        "要启用完整的细胞通讯分析，请：\n\n"
        "  [CellChat -- R 包]:\n"
        f"  1. 确认 R ({RSCRIPT_BIN}) 已安装\n"
        "  2. 安装 CellChat R 包:\n"
        "     R -e 'install.packages(\"remotes\"); "
        "remotes::install_github(\"sqjin/CellChat\")'\n"
        "     依赖项: NMF, Rcpp, RcppArmadillo, circlize, igraph\n"
        f"  当前错误: {_cellchat_r_error}\n\n"
        "  [CellPhoneDB -- Python 包]:\n"
        "  1. pip install cellphonedb\n"
        "  2. 下载数据库: cellphonedb database download\n"
        f"  当前错误: {_cpdb_error}\n\n"
        "两项均安装并配置后重跑本 notebook 即可。\n"
        "=" * 60
    )
    print(_skip_msg)
elif not _cellchat_ready:
    _skip_msg = (
        "=" * 60 + "\n"
        "CellChat R 环境不可用，CellChat 通讯分析将跳过。\n\n"
        "要启用 CellChat:\n"
        "  1. 确认 R 已安装\n"
        "  2. R -e 'install.packages(\"remotes\"); "
        "remotes::install_github(\"sqjin/CellChat\")'\n"
        f"  当前错误: {_cellchat_r_error}\n\n"
        f"CellPhoneDB (Python) 可用: {'是' if _cpdb_available else '否'}\n"
        "=" * 60
    )
    print(_skip_msg)
elif not _cpdb_available:
    _skip_msg = (
        "=" * 60 + "\n"
        "CellPhoneDB Python 包不可用，将跳过其统计分析。\n\n"
        "要启用 CellPhoneDB:\n"
        "  1. pip install cellphonedb\n"
        "  2. cellphonedb database download\n"
        f"  当前错误: {_cpdb_error}\n\n"
        "CellChat (R) 可用 -- 将正常执行通讯网络分析。\n"
        "=" * 60
    )
    print(_skip_msg)
else:
    print("CellChat (R) 和 CellPhoneDB (Python) 均就绪 —— "
          "将完整运行细胞通讯分析。")


## 1. CellChat -- 配体-受体通讯网络推断 (R subprocess)

**为什么用 CellChat？** CellChat 是目前单细胞领域最广泛使用的
细胞通讯分析工具之一（Jin et al., Nature Communications 2021）。
其核心优势在于：

1. **curated 数据库而非简单共表达**：CellChatDB 手工整理了
   2000+ 配体-受体相互作用关系，涵盖分泌型、细胞表面和
   ECM 三类信号，避免了"只要两个基因分别表达就算通讯"的假阳性问题

2. **质量作用模型 (law of mass action)**：通讯概率不是简单的
   配体表达 x 受体表达，而是基于配体-受体结合动力学的
   数学模型——高亲和力的 LR 对即使表达不高也能得分

3. **网络分析能力**：内置通讯模式识别 (communication pattern)、
   信号通路系统分析（识别主导的 outgoing/incoming 信号）、
   多数据集比较框架

**为什么走 subprocess Rscript 而非 rpy2？** ADR-0007 明确：
CellChat 是 heavy R 工具，rpy2 + anndata2ri 在大对象转换上
脆弱且依赖升级易崩。subprocess 模式更鲁棒：
写 .mtx/.csv → Rscript --vanilla → 读结果。
R 脚本独立可调试，不影响 Python 侧环境。

**数据导出**：CellChat 需要原始计数矩阵（genes x cells 的 .mtz 格式）
和细胞元数据（含细胞类型标签）。同时导出 UMAP embedding 供
CellChat 的可视化复用。

In [ ]:
# === Step 1: 导出 CellChat 输入数据 ===
# CellChat R 包需要原始计数 + 细胞类型标签。
# 数据格式：genes x cells 的 MTX（符合 R 侧 Matrix 读取惯例）。
# 优先使用 counts layer（原始计数），log-normalized 会破坏 CellChat 的
# over-expression 检测（该步骤需要原始分布来判断"高表达"的统计显著性）。

if not _cellchat_ready:
    print("CellChat 不可用 —— 跳过数据导出。")
else:
    shutil.rmtree(CELLCHAT_WORK_DIR, ignore_errors=True)
    os.makedirs(CELLCHAT_WORK_DIR, exist_ok=True)

    # 获取计数矩阵
    if "counts" in adata.layers:
        _X_export = adata.layers["counts"]
        print("使用 counts layer 作为原始计数")
    elif adata.raw is not None:
        _X_export = adata.raw[:, adata.var_names].X
        print("使用 adata.raw.X 作为原始计数")
    else:
        _X_export = adata.X
        print("WARNING: 无 counts layer 和 raw，使用 adata.X（可能已是 log-normalized）")

    # 过滤有效分组 —— 排除 NaN + 小簇
    _group_for_export = adata.obs[GROUP_COL].astype(str)
    _use_mask = _group_for_export.isin(_valid_groups)
    _adata_sub = adata[_use_mask].copy() if not _use_mask.all() else adata

    # 写 .mtz（genes x cells —— CellChat 需要基因在行）
    _X_gc = csr_matrix(_adata_sub.layers["counts"] if "counts" in _adata_sub.layers else _adata_sub.X).T.tocoo()
    _mtx_path = os.path.join(CELLCHAT_WORK_DIR, "counts.mtx")
    mmwrite(_mtx_path, _X_gc)
    print(f"计数矩阵已导出: {_mtx_path}  (shape genes x cells: {_X_gc.shape})")

    # 细胞元数据 —— 核心是 cell_type 标签
    _meta = _adata_sub.obs[[GROUP_COL]].copy()
    _meta.columns = ["cell_type"]
    _meta["cell_type"] = _meta["cell_type"].astype(str)
    _meta["cell_id"] = _adata_sub.obs_names.astype(str)
    _meta = _meta[["cell_id", "cell_type"]]
    _meta_path = os.path.join(CELLCHAT_WORK_DIR, "cell_meta.csv")
    _meta.to_csv(_meta_path, index=False)
    print(f"细胞元数据已导出: {_meta_path}  ({len(_meta)} 细胞)")

    # 基因列表
    _gene_df = pd.DataFrame({
        "gene": _adata_sub.var_names.astype(str),
    })
    _gene_path = os.path.join(CELLCHAT_WORK_DIR, "genes.csv")
    _gene_df.to_csv(_gene_path, index=False)
    print(f"基因列表已导出: {_gene_path}  ({len(_gene_df)} 基因)")

    # UMAP 坐标（让 CellChat 复用 Python 侧的 embedding）
    if "X_umap" in _adata_sub.obsm:
        _umap = pd.DataFrame(
            _adata_sub.obsm["X_umap"][:, :2],
            index=_adata_sub.obs_names.astype(str),
            columns=["UMAP1", "UMAP2"],
        )
        _umap.index.name = "cell_id"
        _umap.reset_index().to_csv(
            os.path.join(CELLCHAT_WORK_DIR, "existing_umap.csv"), index=False
        )
        print("UMAP 坐标已导出")

    del _adata_sub, _X_gc
    gc.collect()


> **关于 `C` 前缀标签**：CellChat 内部会拒收纯数字字符型标签（如 `"0"`），
> 因此本 notebook 在导出细胞元数据时自动对 `cell_type` 标签添加 `C` 前缀
> （`paste0("C", cell_meta$cell_type)`）。下游 CSV 输出中 `C0` 对应
> `adata.obs` 中的 leiden cluster `0`，`C1` 对应 cluster `1`，依此类推。
> 跨表交叉比对时请注意此映射关系。

In [ ]:
# === Step 2: 生成并运行 CellChat R 脚本 ===
# CellChat R 脚本逻辑（重写自 CellChat 官方 vignette + CD4 deep-analysis
# LR 分析经验，按本项目规范简化）：
# 1. 读入数据 → 创建 CellChat 对象 (CellChatDB.human)
# 2. 预处理：identifyOverExpressedGenes + identifyOverExpressedInteractions
# 3. 计算通讯概率：computeCommunProb（population.size=TRUE 考虑细胞群大小差异）
# 4. 过滤低质量通讯：filterCommunication（min.cells=10）
# 5. 聚合网络：aggregateNet
# 6. 导出结果表格 → CSV

if not _cellchat_ready:
    print("CellChat 不可用 —— 跳过 R 脚本运行。")
else:
    _r_script = os.path.join(CELLCHAT_WORK_DIR, "run_cellchat.R")
    _r_code = r'''#!/usr/bin/env Rscript
suppressPackageStartupMessages({
  library(CellChat)
  library(Matrix)
  library(data.table)
  library(dplyr)
  library(ggplot2)
})

args <- commandArgs(trailingOnly = TRUE)
input_mtx    <- args[1]
input_meta   <- args[2]
input_gene   <- args[3]
input_umap   <- args[4]
output_prefix<- args[5]
db_species   <- args[6]

# ---- 1. 读入数据 ----
message("Loading data...")
expr <- readMM(input_mtx)
rownames(expr) <- fread(input_gene, data.table = FALSE)$gene
colnames(expr) <- fread(input_meta, data.table = FALSE)$cell_id

cell_meta <- fread(input_meta, data.table = FALSE)
rownames(cell_meta) <- cell_meta$cell_id
cell_meta$cell_type <- as.factor(paste0("C", cell_meta$cell_type))

# ---- 2. 选数据库 ----
if (db_species == "mouse") {
  CellChatDB <- CellChatDB.mouse
} else {
  CellChatDB <- CellChatDB.human
}
# 使用全部类别（Secreted + Cell-Cell Contact + ECM-Receptor）
CellChatDB.use <- CellChatDB

# ---- 3. 创建 CellChat 对象 ----
message("Creating CellChat object...")
cellchat <- createCellChat(object = expr, meta = cell_meta, group.by = "cell_type")
cellchat@DB <- CellChatDB.use

# ---- 4. 预处理 ----
message("Preprocessing...")
cellchat <- subsetData(cellchat)
future::plan("multisession", workers = 1)
cellchat <- identifyOverExpressedGenes(cellchat, do.fast = FALSE)
cellchat <- identifyOverExpressedInteractions(cellchat)

# ---- 5. 计算通讯概率 ----
message("Computing communication probability...")
cellchat <- computeCommunProb(cellchat, population.size = TRUE)
cellchat <- filterCommunication(cellchat, min.cells = 10)

# ---- 6. 聚合网络 ----
message("Aggregating networks...")
cellchat <- aggregateNet(cellchat)
tryCatch({ cellchat <- netAnalysis_computeCentrality(cellchat, slot.name = "netP") }, error = function(e) { message("Centrality computation skipped: ", e$message) })

# ---- 7. 导出结果表格 ----
message("Exporting results...")

# 7a. 通讯概率（通路级别 netP）
netP <- subsetCommunication(cellchat, slot.name = "netP")
write.csv(netP, paste0(output_prefix, "_netP.csv"), row.names = FALSE)

# 7b. 通讯概率（相互作用级别 net）
net <- subsetCommunication(cellchat)
write.csv(net, paste0(output_prefix, "_net.csv"), row.names = FALSE)

# 7c. 聚合计数网络
# outdegree（发送者）-- 每群细胞作为信号源的强度
outdeg <- tryCatch(as.data.frame(cellchat@net$count[, , "out"]), error = function(e) { message("outdeg skipped: ", e$message); data.frame() })
write.csv(outdeg, paste0(output_prefix, "_outdeg.csv"))

# indegree（接收者）-- 每群细胞作为信号靶的强度
indeg <- tryCatch(as.data.frame(cellchat@net$count[, , "in"]), error = function(e) { message("indeg skipped: ", e$message); data.frame() })
write.csv(indeg, paste0(output_prefix, "_indeg.csv"))

# 7d. 网络中心性（保卫：上游 netAnalysis_computeCentrality 已 tryCatch，
# 小数据集失败时 centr 槽位为 NULL，需防 as.data.frame(NULL) 崩）
centr_slot <- slot(cellchat, "netP")[["centr"]]
if (!is.null(centr_slot)) {
  net_centr <- as.data.frame(centr_slot)
  write.csv(net_centr, paste0(output_prefix, "_centrality.csv"))
} else {
  message("centrality.csv skipped: centrality not computed")
}

# ---- 8. 可视化 ----
message("Generating figures...")

# Fig A: 相互作用计数概览（circle plot 准备）
png(paste0(output_prefix, "_interaction_counts.png"),
    width = 2000, height = 1800, res = 200)
print(netVisual_circle(cellchat@net$count,
                        weight.scale = TRUE, label.edge = FALSE,
                        title.name = "Interaction counts"))
dev.off()

# Fig B: 通路热图（依赖 centrality，上游失败时跳过）
tryCatch({
  png(paste0(output_prefix, "_pathway_heatmap.png"),
      width = 2200, height = 2000, res = 200)
  print(netAnalysis_signalingRole_heatmap(cellchat, pattern = "outgoing",
                                          title = "Outgoing signaling patterns"))
  dev.off()
}, error = function(e) {
  message("pathway_heatmap.png skipped: ", e$message)
})

# Fig C: 通路贡献气泡图（依赖 centrality，上游失败时跳过）
tryCatch({
  png(paste0(output_prefix, "_pathway_bubble.png"),
      width = 2400, height = 2000, res = 200)
  print(netAnalysis_signalingRole_scatter(cellchat,
                                          title = "Signaling role (sender vs receiver)"))
  dev.off()
}, error = function(e) {
  message("pathway_bubble.png skipped: ", e$message)
})

message("CellChat completed successfully.")
'''

    # 写 R 脚本
    with open(_r_script, "w", encoding="utf-8") as _f:
        _f.write(_r_code)
    print(f"R 脚本已生成: {_r_script}")

    # 调用 Rscript
    _mtx_path  = os.path.join(CELLCHAT_WORK_DIR, "counts.mtx")
    _meta_path = os.path.join(CELLCHAT_WORK_DIR, "cell_meta.csv")
    _gene_path = os.path.join(CELLCHAT_WORK_DIR, "genes.csv")
    _umap_path = os.path.join(CELLCHAT_WORK_DIR, "existing_umap.csv")
    _out_prefix = os.path.join(CELLCHAT_WORK_DIR, "cellchat")

    _cmd = [
        RSCRIPT_BIN, "--vanilla", _r_script,
        _mtx_path, _meta_path, _gene_path, _umap_path, _out_prefix,
        CELLCHAT_DB_SPECIES,
    ]
    print(f"正在运行: {' '.join(_cmd)}")

    _env = os.environ.copy()
    _env["R_PROFILE_USER"] = ""
    _env["R_ENVIRON_USER"] = ""

    try:
        _res = subprocess.run(
            _cmd, capture_output=True, text=True,
            env=_env, timeout=1800,  # CellChat 可能需要较长时间
        )
        _stdout_path = os.path.join(CELLCHAT_WORK_DIR, "stdout.log")
        _stderr_path = os.path.join(CELLCHAT_WORK_DIR, "stderr.log")
        with open(_stdout_path, "w") as _f:
            _f.write(_res.stdout or "")
        with open(_stderr_path, "w") as _f:
            _f.write(_res.stderr or "")

        if _res.returncode != 0:
            print(f"CellChat 运行失败 (exitcode={_res.returncode})")
            print(f"  STDOUT: {_stdout_path}")
            print(f"  STDERR: {_stderr_path}")
            print(f"  STDERR 尾部: {(_res.stderr or '')[-500:]}")
        else:
            print("CellChat 运行成功")
            _cellchat_done = True
    except subprocess.TimeoutExpired:
        print("CellChat 运行超时（>30 分钟），已终止")


In [ ]:
# === Step 3: 读回 CellChat 结果 ===
# 读取 CellChat R 产出的 CSV 表格和图片。

_cellchat_results = {}
if _cellchat_ready and _cellchat_done:
    _out_prefix = os.path.join(CELLCHAT_WORK_DIR, "cellchat")

    # 读回通讯概率表
    for _suffix, _label in [
        ("_netP.csv", "通路级通讯概率 (netP)"),
        ("_net.csv", "相互作用级通讯概率 (net)"),
        ("_outdeg.csv", "发送者出度 (outdegree)"),
        ("_indeg.csv", "接收者入度 (indegree)"),
        ("_centrality.csv", "网络中心性"),
    ]:
        _path = _out_prefix + _suffix
        if os.path.exists(_path):
            _df = pd.read_csv(_path)
            _cellchat_results[_label] = _df
            print(f"已加载: {_label}  ({len(_df)} 行)")
        else:
            print(f"未找到: {_path}")

    # 复制 R 产出的图片到 results/figures/
    for _fname in [
        "cellchat_interaction_counts.png",
        "cellchat_pathway_heatmap.png",
        "cellchat_pathway_bubble.png",
    ]:
        _src = os.path.join(CELLCHAT_WORK_DIR, _fname)
        if os.path.exists(_src):
            _dst = os.path.join("results", "figures", f"stage7_{_fname}")
            shutil.copy2(_src, _dst)
            print(f"CellChat figure 已复制: {_dst}")

    # 写入 adata.uns
    if _cellchat_results:
        # 保存通路级结果摘要
        _netp_df = _cellchat_results.get("通路级通讯概率 (netP)")
        if _netp_df is not None:
            _netp_csv = "results/tables/stage7_cellchat_netP.csv"
            _netp_df.to_csv(_netp_csv, index=False)
            print(f"CellChat netP 已保存: {_netp_csv}")
            # 简要摘要
            if "pathway_name" in _netp_df.columns:
                _top_pathways = _netp_df["pathway_name"].value_counts().head(10)
                print(f"\nTop 10 被检测到的信号通路:")
                for _pw, _cnt in _top_pathways.items():
                    print(f"  {_pw}: {_cnt} 对 LR")

        _net_df = _cellchat_results.get("相互作用级通讯概率 (net)")
        if _net_df is not None:
            _net_csv = "results/tables/stage7_cellchat_net.csv"
            _net_df.to_csv(_net_csv, index=False)
            print(f"CellChat net 已保存: {_net_csv}")
else:
    print("CellChat 结果不可用 —— 无数据读回。"
          "安装 CellChat R 包后重跑本 notebook 即可。")


## 2. CellPhoneDB —— 配体-受体显著性分析 (Python)

**为什么用 CellPhoneDB？** CellPhoneDB (Efremova et al., Nature Protocols
2020) 在配体-受体分析中有独特优势：

1. **多亚基复合物建模**：不同于大多数工具将"配体"和"受体"
   视为单个基因，CellPhoneDB 将蛋白质复合物作为互作单位。
   例如 IL-12 是由 IL12A + IL12B 组成的异二聚体，
   IL-12 受体是 IL12RB1 + IL12RB2 —— CellPhoneDB 正确建模这种
   多对多关系，避免了简化处理的假阴性

2. **统计显著性**：通过置换检验 (permutation test) 评估每个
   LR 对是否显著高于随机背景，提供 p-value 用于多重检验校正

3. **curated 数据库**：手工整理配体-受体-复合物关系，
   特别考虑了 heteromeric 受体（如 IL-2 受体 alpha/beta/gamma 三元复合物）

**工作流程**：
1. 导出表达矩阵 (counts.txt) + 细胞元数据 (meta.txt)
2. 调用 `cellphonedb method statistical_analysis`
3. 解读显著 LR 对及其生物学意义

当前环境中 CellPhoneDB **未安装**（通过守卫检测），此步骤将跳过。
安装后（`pip install cellphonedb` + 数据库下载）重跑即可启用。

In [ ]:
# === CellPhoneDB 配体-受体显著性分析 ===
# CellPhoneDB 需要原始计数矩阵 + 细胞类型元数据作为输入。
# 当前环境中 CellPhoneDB Python 包不可用，跳过分析。

if not _cpdb_available:
    print("CellPhoneDB 不可用 —— 跳过显著性分析。")
    print("安装指引:")
    print("  1. pip install cellphonedb")
    print("  2. cellphonedb database download")
    print("安装完成后重跑本 notebook 即可启用此步骤。")
else:
    # CellPhoneDB 可用 —— 准备输入并运行
    import cellphonedb  # noqa: F811

    _cpdb_dir = "results/_cellphonedb_tmp"
    shutil.rmtree(_cpdb_dir, ignore_errors=True)
    os.makedirs(_cpdb_dir, exist_ok=True)

    # 准备 counts.txt（genes x cells，tab 分隔）
    if "counts" in adata.layers:
        _X_cpdb = adata.layers["counts"]
    elif adata.raw is not None:
        _X_cpdb = adata.raw[:, adata.var_names].X
    else:
        _X_cpdb = adata.X

    _counts_df = pd.DataFrame(
        _X_cpdb.toarray() if sp.issparse(_X_cpdb) else np.asarray(_X_cpdb),
        index=adata.obs_names.astype(str),
        columns=adata.var_names.astype(str),
    ).T  # genes x cells
    _counts_path = os.path.join(_cpdb_dir, "counts.txt")
    _counts_df.to_csv(_counts_path, sep="\t")
    print(f"表达矩阵已导出: {_counts_path}  ({_counts_df.shape[0]} genes x {_counts_df.shape[1]} cells)")

    # 准备 meta.txt（cell -> cell_type 映射）
    _meta_cpdb = pd.DataFrame({
        "Cell": adata.obs_names.astype(str),
        "cell_type": adata.obs[GROUP_COL].astype(str),
    })
    _meta_path = os.path.join(_cpdb_dir, "meta.txt")
    _meta_cpdb.to_csv(_meta_path, sep="\t", index=False)
    print(f"元数据已导出: {_meta_path}")

    # 运行 CellPhoneDB statistical_analysis
    _cpdb_cmd = [
        "cellphonedb", "method", "statistical_analysis",
        _meta_path, _counts_path,
        "--output-path", _cpdb_dir,
        "--pvalue", str(CPDB_PVAL_THRESHOLD),
        "--threads", "4",
    ]
    print(f"正在运行: {' '.join(_cpdb_cmd)}")

    try:
        _res = subprocess.run(
            _cpdb_cmd, capture_output=True, text=True, timeout=3600,
        )
        if _res.returncode != 0:
            print(f"CellPhoneDB 运行失败 (exitcode={_res.returncode})")
            print(f"  STDERR: {_res.stderr[:500]}")
        else:
            # 读回显著结果
            _sig_means = os.path.join(_cpdb_dir, "significant_means.txt")
            if os.path.exists(_sig_means):
                _sig_df = pd.read_csv(_sig_means, sep="\t")
                print(f"显著 LR pairs 数: {len(_sig_df)}")
                _sig_csv = "results/tables/stage7_cpdb_significant_means.csv"
                _sig_df.to_csv(_sig_csv, index=False)
                print(f"已保存: {_sig_csv}")
                _cpdb_done = True
            else:
                print("WARNING: significant_means.txt 未生成")
    except subprocess.TimeoutExpired:
        print("CellPhoneDB 运行超时（>60 分钟），已终止")


## 3. 配体-受体表达概览 (纯 Python, 无需外部工具)

**为什么需要这个部分？** 当 CellChat 和 CellPhoneDB 都不可用时，
仍需要一种方式快速了解"哪些配体-受体对在数据中存在表达"。
此模块从表达数据中直接提取配体和受体的平均表达量，
不需要任何外部数据库或 R 环境。

**方法**：对每个细胞类型分组：
1. 计算每个基因的平均表达量
2. 在数据集中搜索常见的配体-受体基因对（基于基因名正则匹配）
3. 如果数据中存在配体基因在群 A 高表达、受体基因在群 B 高表达，
   则提示可能存在 A → B 的通讯

**局限性**：这只是一个基于表达模式的快速筛查，
**不是** CellChat/CellPhoneDB 的替代品。真正的细胞通讯推断
需要 curated LR 数据库 + 统计模型（质量作用/置换检验）。
方法学论文中不宜仅用此方法。

本 cell 始终执行，不依赖任何外部工具。

In [ ]:
# === 配体-受体表达概览 ===
# 纯 Python 实现，基于表达矩阵直接搜索配体/受体基因，
# 计算每群细胞的平均表达量。
# 这不是 CellChat/CellPhoneDB 的替代品——它只是一个"表达层面"的快速筛查。
# 真实的通讯推断需要 curated LR 数据库和统计模型。

print("=== 配体-受体表达概览 ===\n")
print("（此方法仅基于表达量快速筛查，非正式的通讯推断。")
print(" 正式分析请安装 CellChat 或 CellPhoneDB。）\n")

# 获取表达矩阵（log-normalized 用于比较均值）
_X_lr = adata.X
_group_labels = adata.obs[GROUP_COL].astype(str)

# 获取 var_names（基因符号）
_gene_names = adata.var_names.astype(str).tolist()

# 常见配体-受体对列表（手工整理，作为示例）
# PI 可根据具体研究问题扩展此列表
_LR_PAIRS = [
    # 生长因子通路
    ("EGF", "EGFR"), ("TGFA", "EGFR"), ("AREG", "EGFR"),
    ("HGF", "MET"), ("FGF7", "FGFR2"), ("FGF10", "FGFR2"),
    ("IGF1", "IGF1R"),
    # WNT 通路
    ("WNT2", "FZD1"), ("WNT5A", "FZD5"), ("RSPO3", "LGR5"),
    # Notch 通路
    ("JAG1", "NOTCH1"), ("DLL1", "NOTCH1"), ("DLL4", "NOTCH1"),
    # BMP/TGF-beta 通路
    ("BMP2", "BMPR1A"), ("BMP4", "BMPR2"), ("TGFB1", "TGFBR1"),
    # 趋化因子
    ("CXCL12", "CXCR4"), ("CCL5", "CCR5"), ("CCL20", "CCR6"),
    ("CXCL8", "CXCR1"), ("CXCL10", "CXCR3"),
    # 炎症/细胞因子
    ("IL1B", "IL1R1"), ("IL6", "IL6R"), ("TNF", "TNFRSF1A"),
    ("IFNG", "IFNGR1"), ("IL10", "IL10RA"),
    # 免疫检查点
    ("CD274", "PDCD1"), ("PDCD1LG2", "PDCD1"),  # PD-L1/L2 → PD-1
    ("CD80", "CTLA4"), ("CD86", "CTLA4"),
    # 黏附/迁移
    ("ICAM1", "ITGAL"), ("VCAM1", "ITGA4"),
    # 胃上皮相关
    ("GAST", "CCKBR"), ("TFF1", "CXCR4"), ("MUC1", "TLR4"),
]

# 检查哪些 LR 对在数据集中（配体和受体基因都存在）
_valid_pairs = []
for _lig, _rec in _LR_PAIRS:
    _lig_in = _lig in _gene_names
    _rec_in = _rec in _gene_names
    if _lig_in and _rec_in:
        _valid_pairs.append((_lig, _rec))

_lig_genes = sorted(set(p[0] for p in _valid_pairs))
_rec_genes = sorted(set(p[1] for p in _valid_pairs))
print(f"配体-受体对: {len(_valid_pairs)}/{len(_LR_PAIRS)} 对在数据集中存在")
print(f"  配体基因: {len(_lig_genes)} 个")
print(f"  受体基因: {len(_rec_genes)} 个")

if len(_valid_pairs) == 0:
    print("无配体-受体基因对存在于数据集中 —— 跳过表达概览。")
else:
    # 获取配体和受体基因的表达量（log-normalized）
    _lig_expr = pd.DataFrame(
        adata[:, _lig_genes].X.toarray() if sp.issparse(adata[:, _lig_genes].X)
        else np.asarray(adata[:, _lig_genes].X),
        index=adata.obs_names,
        columns=_lig_genes,
    )
    _rec_expr = pd.DataFrame(
        adata[:, _rec_genes].X.toarray() if sp.issparse(adata[:, _rec_genes].X)
        else np.asarray(adata[:, _rec_genes].X),
        index=adata.obs_names,
        columns=_rec_genes,
    )

    # 按细胞群计算平均表达
    _lig_mean = _lig_expr.groupby(_group_labels).mean()
    _rec_mean = _rec_expr.groupby(_group_labels).mean()

    # 对每个有效 LR 对，计算"通讯强度" = 配体在发送者群中的平均表达
    # × 受体在接收者群中的平均表达。这不是统计推断，只是表达层面的启发式指标。
    _lr_scores = {}
    for _lig, _rec in _valid_pairs:
        for _sender in _lig_mean.index:
            for _receiver in _rec_mean.index:
                _l_expr = _lig_mean.loc[_sender, _lig]
                _r_expr = _rec_mean.loc[_receiver, _rec]
                _score = float(_l_expr) * float(_r_expr)
                if _score > 0:
                    _lr_scores[(_lig, _rec, _sender, _receiver)] = _score

    if _lr_scores:
        _lr_df = pd.DataFrame([
            {"ligand": k[0], "receptor": k[1],
             "sender": k[2], "receiver": k[3],
             "expr_product": v}
            for k, v in _lr_scores.items()
        ]).sort_values("expr_product", ascending=False)

        print(f"\n共计算 {len(_lr_df):,} 条 cell-type → cell-type 链接")
        print(f"表达积范围: [{_lr_df['expr_product'].min():.4f}, "
              f"{_lr_df['expr_product'].max():.4f}]")
        print(f"\nTop 10 高表达积 LR 对 (sender → receiver):")
        _top10 = _lr_df.head(10)
        for _, _row in _top10.iterrows():
            print(f"  {_row['ligand']} → {_row['receptor']}: "
                  f"{_row['sender']} → {_row['receiver']} "
                  f"(expr_product={_row['expr_product']:.4f})")

        # 保存
        _lr_csv = "results/tables/stage7_lr_expr_overview.csv"
        _lr_df.to_csv(_lr_csv, index=False)
        print(f"\nLR 表达概览已保存: {_lr_csv}")
        _lr_overview_done = True
    else:
        print("所有 LR 对在各群中的表达积为 0 —— 无数据可展示。")


## 4. 通讯网络可视化

可视化分为三部分：

### 4a. 配体-受体表达 Dotplot（总是产出）
灵感来自 student-code CD4 deep-analysis 的 `LR_AXES` 方案：
对已整理好的配体-受体对，用 dotplot 同时展示配体（发送者群）
和受体（接收者群）的表达量，一目了然地看到哪些 LR 对在
哪些细胞群中活跃。

### 4b. CellChat 通讯 circle plot（CellChat 完成后产出）
用 networkx + matplotlib 绘制 circle plot，展示细胞群之间的
通讯流向（箭头 + 线宽表示通讯强度）。

### 4c. CellChat 通讯 heatmap（CellChat 完成后产出）
用 seaborn heatmap 展示 sender-receiver 矩阵中的通讯强度。

以下可视化在有数据时正常产出，无数据时优雅跳过。

In [ ]:
# === 4a. LR Expression Dotplot（始终产出） ===
# 灵感：student-code CD4 deep-analysis LR_AXES dotplot，
# 同时展示配体和受体在各细胞群中的表达模式。
# 横轴：配体基因@发送者群  受体基因@接收者群
# 纵轴：LR 对（配体 → 受体）

if not _lr_overview_done:
    print("LR 表达概览未完成 —— 跳过 dotplot。")
else:
    _top_n = min(N_TOP_LR, len(_lr_df))
    _top_pairs = _lr_df.head(_top_n)

    # 构建 dotplot 数据：对每个 top LR 对，列出配体在各群 + 受体在各群的表达
    _groups_sorted = sorted(_valid_groups, key=lambda x: (
        0 if x == "nan" else 1, x
    ))
    _plot_data = []
    for _, _row in _top_pairs.iterrows():
        _lig = _row["ligand"]
        _rec = _row["receptor"]
        _pair_label = f"{_lig} → {_rec}"
        for _g in _groups_sorted:
            _l_val = _lig_mean.loc[_g, _lig] if _g in _lig_mean.index else np.nan
            _r_val = _rec_mean.loc[_g, _rec] if _g in _rec_mean.index else np.nan
            _plot_data.append({
                "pair": _pair_label,
                "group": _g,
                "ligand_expr": _l_val,
                "receptor_expr": _r_val,
            })
    _plot_df = pd.DataFrame(_plot_data)

    # 画 dotplot
    _n_pairs = _top_pairs["ligand"].nunique() if "ligand" in _top_pairs.columns else len(_plot_df["pair"].unique())
    _n_grps = len(_groups_sorted)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(max(10, _n_grps * 0.8),
                                                    max(6, _n_pairs * 0.35)))

    # 左图：配体表达
    _lig_pivot = _plot_df.pivot_table(
        index="pair", columns="group", values="ligand_expr", aggfunc="mean"
    )
    _lig_pivot = _lig_pivot.reindex(
        index=[f"{r['ligand']} → {r['receptor']}" for _, r in _top_pairs.iterrows()]
        if len(_top_pairs) > 0 else _lig_pivot.index
    ).dropna(how="all")
    if len(_lig_pivot) > 0:
        sns.heatmap(
            _lig_pivot, cmap="Reds", ax=ax1,
            xticklabels=True, yticklabels=True,
            linewidths=0.3, cbar_kws={"label": "配体平均表达 (log)"},
        )
    ax1.set_title("配体表达量（按细胞群）")
    ax1.set_xlabel("细胞群（潜在发送者）")
    ax1.set_ylabel("配体-受体对")

    # 右图：受体表达
    _rec_pivot = _plot_df.pivot_table(
        index="pair", columns="group", values="receptor_expr", aggfunc="mean"
    )
    _rec_pivot = _rec_pivot.reindex(
        index=[f"{r['ligand']} → {r['receptor']}" for _, r in _top_pairs.iterrows()]
        if len(_top_pairs) > 0 else _rec_pivot.index
    ).dropna(how="all")
    if len(_rec_pivot) > 0:
        sns.heatmap(
            _rec_pivot, cmap="Blues", ax=ax2,
            xticklabels=True, yticklabels=True,
            linewidths=0.3, cbar_kws={"label": "受体平均表达 (log)"},
        )
    ax2.set_title("受体表达量（按细胞群）")
    ax2.set_xlabel("细胞群（潜在接收者）")
    ax2.set_ylabel("配体-受体对")

    plt.tight_layout()
    _lr_dotplot_path = "results/figures/stage7_lr_expr_dotplot.png"
    fig.savefig(_lr_dotplot_path, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"LR 表达 dotplot 已保存: {_lr_dotplot_path}")


In [ ]:
# === 4b. CellChat 通讯网络可视化（CellChat 完成后产出） ===

_cellchat_has_netP = (
    _cellchat_done
    and any("通路级通讯概率" in k for k in _cellchat_results)
)

if _cellchat_has_netP:
    _netp = _cellchat_results.get("通路级通讯概率 (netP)")
    if _netp is not None and len(_netp) > 0:
        # 聚合 sender-receiver 交互计数
        if {"source", "target"}.issubset(_netp.columns):
            _sr_counts = _netp.groupby(["source", "target"]).size().reset_index(name="count")
            _sr_pivot = _sr_counts.pivot_table(
                index="source", columns="target", values="count", fill_value=0
            )

            # Sender-Receiver Heatmap
            if len(_sr_pivot) > 0:
                fig, ax = plt.subplots(
                    figsize=(max(6, len(_sr_pivot.columns) * 0.7),
                             max(5, len(_sr_pivot.index) * 0.4))
                )
                sns.heatmap(
                    _sr_pivot, cmap="YlOrRd", ax=ax,
                    annot=True, fmt="d", linewidths=0.5,
                    xticklabels=True, yticklabels=True,
                    cbar_kws={"label": "显著 LR 交互数量"},
                )
                ax.set_title("CellChat: 发送者 → 接收者 通讯交互计数")
                ax.set_xlabel("接收者")
                ax.set_ylabel("发送者")
                plt.tight_layout()
                _heatmap_path = "results/figures/stage7_cellchat_sr_heatmap.png"
                fig.savefig(_heatmap_path, dpi=200, bbox_inches="tight")
                plt.close("all")
                print(f"CellChat S-R heatmap 已保存: {_heatmap_path}")

            # Circle plot -- 使用 networkx + matplotlib 替代 circlize
            try:
                import networkx as nx

                _G = nx.DiGraph()
                for _, _row in _sr_counts.iterrows():
                    _G.add_edge(_row["source"], _row["target"], weight=_row["count"])

                if _G.number_of_edges() > 0:
                    fig, ax = plt.subplots(figsize=(10, 10))
                    _pos = nx.circular_layout(_G)
                    _weights = [_G[u][v]["weight"] for u, v in _G.edges()]
                    _max_w = max(_weights) if _weights else 1
                    _widths = [0.5 + 5.0 * w / _max_w for w in _weights]

                    nx.draw_networkx_nodes(
                        _G, _pos, ax=ax,
                        node_size=1200, node_color="lightblue",
                        edgecolors="black", linewidths=1,
                    )
                    nx.draw_networkx_edges(
                        _G, _pos, ax=ax,
                        width=_widths, alpha=0.6,
                        edge_color="grey",
                        connectionstyle="arc3,rad=0.15",
                        arrows=True, arrowsize=20,
                    )
                    nx.draw_networkx_labels(
                        _G, _pos, ax=ax,
                        font_size=9, font_weight="bold",
                    )
                    ax.set_title("CellChat: 细胞间通讯网络")
                    ax.axis("off")
                    plt.tight_layout()
                    _circle_path = "results/figures/stage7_cellchat_circle.png"
                    fig.savefig(_circle_path, dpi=200, bbox_inches="tight")
                    plt.close("all")
                    print(f"CellChat circle plot 已保存: {_circle_path}")
            except ImportError:
                print("networkx 未安装 —— 跳过 circle plot。")
                print("  pip install networkx 可启用此可视化。")
        else:
            print("CellChat netP 表格格式与预期不符 —— 跳过网络可视化。")
    else:
        print("CellChat netP 为空 —— 无可用于可视化的数据。")
else:
    print("CellChat 结果不可用 —— 跳过通讯网络可视化。")
    print("安装 CellChat R 包后重跑即可产出通讯网络图。")


## 运行摘要与元数据

In [ ]:
# 运行摘要与元数据写入 adata.uns。
import datetime as _dt

_stage7_cc_uns = {
    "method": "CellChat (R subprocess) + CellPhoneDB (Python) + LR expression overview",
    "group_col": GROUP_COL,
    "cellchat_r_ready": _cellchat_ready,
    "cellchat_ran": _cellchat_done,
    "cpdb_available": _cpdb_available,
    "cpdb_ran": _cpdb_done,
    "lr_overview_done": _lr_overview_done,
    "timestamp": _dt.datetime.now().isoformat(),
}

_stage7_cc_uns["tools"] = {
    "cellchat": {
        "rscript_bin": RSCRIPT_BIN,
        "species": CELLCHAT_DB_SPECIES,
        "available": _cellchat_ready,
        "ran": _cellchat_done,
    },
    "cellphonedb": {
        "available": _cpdb_available,
        "pval_threshold": CPDB_PVAL_THRESHOLD,
        "ran": _cpdb_done,
    },
}

adata.uns["stage7_cell_communication_v1"] = _stage7_cc_uns
print("运行元数据已写入 adata.uns['stage7_cell_communication_v1']")
print()
print("=" * 50)
print("Stage 7 Cell Communication 执行摘要:")
print(f"  细胞通讯分析:")
print(f"  - CellChat (R):     {'已完成' if (_cellchat_ready and _cellchat_done) else '跳过'}"
        f" (R={'Y' if _cellchat_ready else 'N'})")
print(f"  - CellPhoneDB (Py): {'已完成' if (_cpdb_available and _cpdb_done) else '跳过'}"
        f" (pkg={'Y' if _cpdb_available else 'N'})")
print(f"  - LR 表达概览:      {'已完成' if _lr_overview_done else '跳过'}")
print("=" * 50)
print()
print("下一步建议：")
if not _cellchat_ready:
    print("  1. 安装 CellChat R 包:")
    print("     R -e 'remotes::install_github(\"sqjin/CellChat\")'")
if not _cpdb_available:
    print("  2. 安装 CellPhoneDB:")
    print("     pip install cellphonedb")
    print("     cellphonedb database download")
if _cellchat_ready or _cpdb_available:
    print("  3. 安装完成后重跑本 notebook")
print("  4. 在 stage6 完成细胞类型注释后，将 GROUP_COL 改为 cell_type_final_v1")


In [ ]:
# 内存自检 —— 确保 X 没有被误转为 dense。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

# 检查新增数据
_new_uns_keys = [k for k in adata.uns.keys() if k.startswith("stage7_cell_communication")]
print(f"新增 uns keys: {_new_uns_keys}")


In [ ]:
# 统一追踪字段——stage + version（与上游字段合并，保持 stage3-7 命名一致）
adata.uns["stage"] = "stage7_cell_communication"     # 本 stage 标识
adata.uns["version"] = "v1"                            # 与 OUTPUT_PATH 版本号一致
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"
print("追踪字段已写入: stage=%s  version=%s  status=%s"
      % (adata.uns["stage"], adata.uns["version"], adata.uns["status"]))


In [ ]:
# 写出 checkpoint。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")


In [ ]:
# 释放内存。
del adata
gc.collect()
print("内存已释放")
